<a href="https://colab.research.google.com/github/desouki76/Ahmed/blob/main/TwitterFeedback_%2Bve_ve_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Binary Text Classfcaiton [+ve / - Ve for twittes] fille have 5000 +Ve Twiites and file have -Ve Twittes

# Libraires

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk


In [4]:
from numpy import random
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
#from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
#from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
#from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense, SimpleRNN
from sklearn.preprocessing import LabelEncoder

In [5]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tap')
nltk.download('omw-1.4')
nltk.download('twitter_samples')

import nltk

nltk.download('punkt')
nltk.download('punkt_tab')   # <— this is the new one in NLTK 3.9+
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Error loading punkt_tap: Package 'punkt_tap' not found in
[nltk_data]     index
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package twitter_samples to /root/nltk_data...
[nltk_data]   Unzipping corpora/twitter_samples.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is alrea

True

In [6]:
stop_words = stopwords.words('english')  # i call stope of words of english langauge
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

In [7]:
positive_tweets = nltk.corpus.twitter_samples.strings('positive_tweets.json')
# we call class of 5000  postive twittes
negative_tweets = nltk.corpus.twitter_samples.strings('negative_tweets.json')
# we call class of 5000 negtive twittes and we put it in the varable

In [8]:
pd.DataFrame(positive_tweets).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       5000 non-null   object
dtypes: object(1)
memory usage: 39.2+ KB


In [9]:
pd.DataFrame(negative_tweets).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       5000 non-null   object
dtypes: object(1)
memory usage: 39.2+ KB


# Fucaiton to clean twittes

In [10]:
pd.DataFrame(positive_tweets).head()


,0
0,#FollowFriday @France_Inte @PKuchly57 @Milipol...
1,@Lamb2ja Hey James! How odd :/ Please call our...
2,@DespiteOfficial we had a listen last night :)...
3,@97sides CONGRATS :)
4,yeaaaah yippppy!!! my accnt verified rqst has...


In [11]:
#function to clean twittes
def clean_text(tweet):
    tweet=re.sub(r'http\S+','',tweet) #Plass, @mohamed remover https \S+ any no white space after http
    tweet=re.sub(r'@[A-Za-z0-9_]+','',tweet) #
    tweet=re.sub(r'[^\w\s]','',tweet)
    tweet=re.sub(r'\d+','',tweet)
    tweet=re.sub(r'\s+',' ',tweet)
    tweet=re.sub(r'\s+[a-zA-Z]\s+',' ',tweet)
    tweet=re.sub(r'^[a-zA-Z]\s+',' ',tweet)
    tweet=re.sub(r'\s$',' ',tweet)

    return tweet

In [12]:
# creat function to clean tweets and pre preoss as NLP
def preprocess_tweet(tweets):
    clean_tweets=[]
    for tweet in tweets:
      clean_tweet = clean_text(tweet)
      clean_tweet = nltk.word_tokenize(clean_tweet)
      #tweet_tokens=word_tokenize(tweet)
      #clean_tweet=[word for word in clean_tweet if word not in stop_words]
      #clean_tweet=[lemmatizer.lemmatize(word) for word in clean_tweet]

      #apply "lower case" on all tweets if the word.lower() not in stop words
      clean_tweet =[word.lower() for word in clean_tweet if word.lower() not in stop_words]
      #Apply stemmer رد الكلمه الى اصلها
      clean_tweet=[stemmer.stem(word) for word in clean_tweet]
      clean_tweet=[lemmatizer.lemmatize(word) for word in clean_tweet]
      clean_tweet = [word for word in clean_tweet if len(word) >= 3]
      #delet all words that is 2 letters only
      clean_tweets.append(clean_tweet)
    return clean_tweets



In [13]:
#Apply the prvouis fucation pre process (clean, stop words, lower case, limtizer Port stemmer)
positive_tweets_1=preprocess_tweet(positive_tweets)
negative_tweets_1=preprocess_tweet(negative_tweets)

now tweets are clean lower case limitzed Portstemmed toknized to words


In [14]:

# import sys
# sys.setrecursionlimit(10000)  # allow deeper printing if needed

# import numpy as np
# np.set_printoptions(threshold=np.inf)  # show full arrays

# pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_rows', None)

In [15]:
positive_tweets_1

[['followfriday', 'top', 'engag', 'member', 'commun', 'week'],
 ['hey',
  'jame',
  'odd',
  'plea',
  'call',
  'contact',
  'centr',
  'abl',
  'assist',
  'mani',
  'thank'],
 ['listen', 'last', 'night', 'bleed', 'amaz', 'track', 'scotland'],
 ['congrat'],
 ['yeaaaah',
  'yippppi',
  'accnt',
  'verifi',
  'rqst',
  'succeed',
  'got',
  'blue',
  'tick',
  'mark',
  'profil',
  'day'],
 ['one', 'irresist', 'flipkartfashionfriday'],
 ['dont',
  'like',
  'keep',
  'love',
  'custom',
  'wait',
  'long',
  'hope',
  'enjoy',
  'happi',
  'friday',
  'lwwf'],
 ['second',
  'thought',
  'there',
  'enough',
  'time',
  'new',
  'short',
  'enter',
  'system',
  'sheep',
  'must',
  'buy'],
 ['jgh', 'bayan', 'bye'],
 ['act',
  'mischiev',
  'call',
  'etl',
  'layer',
  'inhous',
  'wareh',
  'app',
  'katamari',
  'well',
  'name',
  'impli'],
 ['followfriday', 'top', 'influenc', 'commun', 'week'],
 ['wouldnt', 'love', 'bigjuicyselfi'],
 ['follow', 'amp', 'follow', 'back'],
 ['perfect'

In [16]:
for tweet in positive_tweets_1[:1000]:
    print(tweet)

['followfriday', 'top', 'engag', 'member', 'commun', 'week']
['hey', 'jame', 'odd', 'plea', 'call', 'contact', 'centr', 'abl', 'assist', 'mani', 'thank']
['listen', 'last', 'night', 'bleed', 'amaz', 'track', 'scotland']
['congrat']
['yeaaaah', 'yippppi', 'accnt', 'verifi', 'rqst', 'succeed', 'got', 'blue', 'tick', 'mark', 'profil', 'day']
['one', 'irresist', 'flipkartfashionfriday']
['dont', 'like', 'keep', 'love', 'custom', 'wait', 'long', 'hope', 'enjoy', 'happi', 'friday', 'lwwf']
['second', 'thought', 'there', 'enough', 'time', 'new', 'short', 'enter', 'system', 'sheep', 'must', 'buy']
['jgh', 'bayan', 'bye']
['act', 'mischiev', 'call', 'etl', 'layer', 'inhous', 'wareh', 'app', 'katamari', 'well', 'name', 'impli']
['followfriday', 'top', 'influenc', 'commun', 'week']
['wouldnt', 'love', 'bigjuicyselfi']
['follow', 'amp', 'follow', 'back']
['perfect', 'alreadi', 'know', 'what', 'wait']
['great', 'new', 'opportun', 'junior', 'triathlet', 'age', 'gatorad', 'seri', 'get', 'entri']
['la

In [17]:
print(positive_tweets_1[110])

['alreadi', 'afternoon', 'let', 'read', 'kahfi', 'day', 'finish']


In [18]:
pd.DataFrame(positive_tweets_1).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       4953 non-null   object
 1   1       4397 non-null   object
 2   2       3693 non-null   object
 3   3       3091 non-null   object
 4   4       2475 non-null   object
 5   5       2011 non-null   object
 6   6       1640 non-null   object
 7   7       1228 non-null   object
 8   8       925 non-null    object
 9   9       641 non-null    object
 10  10      409 non-null    object
 11  11      226 non-null    object
 12  12      118 non-null    object
 13  13      49 non-null     object
 14  14      25 non-null     object
 15  15      11 non-null     object
 16  16      5 non-null      object
 17  17      4 non-null      object
 18  18      2 non-null      object
 19  19      1 non-null      object
dtypes: object(20)
memory usage: 781.4+ KB


In [19]:
pd.DataFrame(negative_tweets_1).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       4934 non-null   object
 1   1       4344 non-null   object
 2   2       3596 non-null   object
 3   3       2855 non-null   object
 4   4       2235 non-null   object
 5   5       1814 non-null   object
 6   6       1443 non-null   object
 7   7       1162 non-null   object
 8   8       905 non-null    object
 9   9       656 non-null    object
 10  10      423 non-null    object
 11  11      268 non-null    object
 12  12      148 non-null    object
 13  13      75 non-null     object
 14  14      31 non-null     object
 15  15      17 non-null     object
 16  16      7 non-null      object
 17  17      4 non-null      object
 18  18      3 non-null      object
 19  19      2 non-null      object
dtypes: object(20)
memory usage: 781.4+ KB


#creat lable and list

In [20]:
#creat list with one coul and 5000 row equal the number of rows of the lsit
positive_lable=[1]*len(positive_tweets_1)
negative_lable=[0]*len(negative_tweets_1)


In [21]:
# we have now +ve tweets and -Ve tweets and we have 0 +ver lable and 1 as -Ve lael
# we need to contacte them
tweets=positive_tweets_1+negative_tweets_1
lables=positive_lable+negative_lable


In [ ]:
#Get vocabulary (unique words)


In [22]:
#lables

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,


In [23]:
#creat table (data fram ) by your self define the coulmn and value to be added
pd.DataFrame({'Scrabed_Tweets':tweets,"O/P_Lable":lables})
#i wanna display 1st 10 rows
pd.DataFrame({'Scrabed_Tweets':tweets,"O/P_Lable":lables}).head(10)

,Scrabed_Tweets,O/P_Lable
0,"[followfriday, top, engag, member, commun, week]",1
1,"[hey, jame, odd, plea, call, contact, centr, a...",1
2,"[listen, last, night, bleed, amaz, track, scot...",1
3,[congrat],1
4,"[yeaaaah, yippppi, accnt, verifi, rqst, succee...",1
5,"[one, irresist, flipkartfashionfriday]",1
6,"[dont, like, keep, love, custom, wait, long, h...",1
7,"[second, thought, there, enough, time, new, sh...",1
8,"[jgh, bayan, bye]",1
9,"[act, mischiev, call, etl, layer, inhous, ware...",1


In [24]:
pd.DataFrame({'Scrabed_Tweets':tweets,"O/P_Lable":lables}).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Scrabed_Tweets  10000 non-null  object
 1   O/P_Lable       10000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 156.4+ KB


In [25]:
zip_data = list(zip(tweets, lables))
random.shuffle(zip_data)
tweets, lables = zip(*zip_data)


In [26]:
pd.DataFrame({'Scrabed_Tweets':tweets,"O/P_Lable":lables}).head(10)


,Scrabed_Tweets,O/P_Lable
0,"[cours, that, understand, notic, issu, well, p...",1
1,"[follow, amp, follow, back]",1
2,"[kik, senight, kik, kikmeboy, gay, teen, amate...",0
3,"[hehe, smile, ear, ear]",1
4,"[haha, know, mess]",1
5,"[mano, hold, hand]",1
6,[],1
7,[thank],1
8,"[glad, sonal, instagram, what]",1
9,"[love, store, best, chocol, select]",1


# Word Embeding
1- Layer Embedidng in DL
2- word2Vec
3- Glove
4- Fasttext

In [36]:
#Embedding
#apply test representaiton by embeding

from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Embedding

In [29]:
#flatten  Flatten into one list of words
all_words=[]
for tweet in tweets:
  for word in tweet:
    all_words.append(word)




In [30]:
#Get vocabulary (unique words)

voc = set (all_words)
voc_size = len(voc)


In [31]:
voc_size

9553

In [43]:

tweets = list(tweets)
labels = list(lables)
# join tokens back into sentence
Onehot_prep = [one_hot (" ".join(tweet), voc_size ) for tweet in tweets]
#the for tweet in tweets coz That’s because keras.preprocessing.text.one_hot()
#works on one string at a time, not on a list of strings.

#Your tweets list doesn’t contain strings, it contains
# lists of tokens (because you already preprocessed them earlier).

In [44]:
print(Onehot_prep[:5])

[[6493, 7872, 83603, 33299, 40695, 63096, 4843, 19170, 77502, 16416], [87165, 47374, 87165, 69396], [41860, 68001, 41860, 74516, 65103, 96544, 85250, 34492, 80452], [33012, 94981, 55517, 55517], [44155, 39289, 88472]]


In [47]:
#apply padding
sent_lenght = 8
embeded_tweets= pad_sequences(Onehot_prep,padding='pre',maxlen=sent_lenght)
#define
#sent length = 8 words
#word vectore repsetnation = 10 for example
#voc_size number of unqe words we have in or our sets

In [48]:
#Apply Model for EMbded
model = Sequential()
model.add(Embedding(voc_size,10,input_length=sent_lenght))
model.compile('adam','mse')


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [49]:
#predect Data
model.predict(embeded_tweets[0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step


array([[ 0.01654765,  0.00160365,  0.03312453,  0.00421169, -0.0009368 ,
         0.02978808, -0.02216253,  0.01327908,  0.04118707,  0.04224483],
       [ 0.00202424,  0.02492621,  0.03635404,  0.04728113,  0.03066598,
         0.02360684,  0.01321826,  0.04675952, -0.0377076 , -0.04331464],
       [-0.03121262,  0.0494622 , -0.02326591,  0.00216863, -0.00988318,
         0.03833586, -0.00462496, -0.00388495,  0.01087673,  0.01475007],
       [ 0.00789427, -0.04348931,  0.00677359,  0.01694337, -0.04157122,
        -0.02871999, -0.00563763, -0.0136417 , -0.002372  , -0.00949639],
       [ 0.02464327,  0.00578577,  0.01725307,  0.01396297, -0.01232159,
        -0.01966715,  0.02420131, -0.01176448,  0.01616028,  0.0440647 ],
       [ 0.02449906, -0.03368508,  0.02541837, -0.01410953,  0.03787932,
        -0.00130128,  0.00453969, -0.01735293, -0.03120352,  0.01992849],
       [ 0.03282446,  0.02600432,  0.03455538, -0.02219766,  0.01211538,
        -0.03542561, -0.03256956,  0.01225996